# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests


# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

hey


In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)


# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [5]:
df_trips.printSchema()
print("Nombre de courses :", df_trips.count())

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

Nombre de courses : 7696617


## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [6]:
from pyspark.sql import functions as F

# Q1: add a unique key to identify each trip
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())
df_trips.cache()  # keep the dataframe in memory to speed up the next queries
df_trips.select("trip_id", "tpep_pickup_datetime", "passenger_count").show(5)

# Q2: trip with the highest passenger count
df_trips.orderBy(F.col("passenger_count").desc()) \
    .select("trip_id", "tpep_pickup_datetime", "passenger_count", "trip_distance", "total_amount") \
    .show(10)

# Q3: average passenger count
df_trips.agg(F.round(F.avg("passenger_count"), 2).alias("avg_passenger_count")).show()


+-----------+--------------------+---------------+
|    trip_id|tpep_pickup_datetime|passenger_count|
+-----------+--------------------+---------------+
|25769803776| 2019-01-01 00:46:40|            1.0|
|25769803777| 2019-01-01 00:59:47|            1.0|
|25769803778| 2018-12-21 13:48:30|            3.0|
|25769803779| 2018-11-28 15:52:25|            5.0|
|25769803780| 2018-11-28 15:56:57|            5.0|
+-----------+--------------------+---------------+
only showing top 5 rows
+-----------+--------------------+---------------+-------------+------------+
|    trip_id|tpep_pickup_datetime|passenger_count|trip_distance|total_amount|
+-----------+--------------------+---------------+-------------+------------+
|25770753732| 2019-01-05 13:12:29|            9.0|          0.0|        12.6|
|25772687771| 2019-01-13 04:13:24|            9.0|          0.0|       12.25|
|25771100063| 2019-01-07 03:19:36|            9.0|          0.0|         9.3|
|25771815874| 2019-01-10 00:43:10|            9.0

In [7]:
#Prep for Q4
# trip duration in minutes (dropoff - pickup)
df_trips = df_trips.withColumn(
    "trip_duration_min",
    F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 2)
)

# keep only trips picked up in January 2019
df_jan = df_trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01") &
    (F.col("tpep_pickup_datetime") < "2019-02-01")
)
df_jan.cache()

cols = ["trip_id", "tpep_pickup_datetime", "trip_distance", "trip_duration_min", "total_amount"]

In [8]:
# Q4a: shortest / longest trips by distance
df_trips.orderBy(F.col("trip_distance").asc()).select(cols).show(5)
df_trips.orderBy(F.col("trip_distance").desc()).select(cols).show(5)

# Q4b: shortest / longest trips by duration
df_trips.orderBy(F.col("trip_duration_min").asc()).select(cols).show(5)
df_trips.orderBy(F.col("trip_duration_min").desc()).select(cols).show(5)

+-----------+--------------------+-------------+-----------------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|trip_duration_min|total_amount|
+-----------+--------------------+-------------+-----------------+------------+
|25769803781| 2018-11-28 16:25:49|          0.0|             2.62|       13.31|
|25769803782| 2018-11-28 16:29:37|          0.0|              4.1|       55.55|
|25769803804| 2019-01-01 00:32:59|          0.0|              0.0|         7.8|
|25769803780| 2018-11-28 15:56:57|          0.0|              1.6|       55.55|
|25769803778| 2018-12-21 13:48:30|          0.0|             4.17|         5.8|
+-----------+--------------------+-------------+-----------------+------------+
only showing top 5 rows
+-----------+--------------------+-------------+-----------------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|trip_duration_min|total_amount|
+-----------+--------------------+-------------+-----------------+------------+
|25775877867| 20

In [9]:
# Q5: number of trips per day (January 2019 only)
trips_per_day = df_jan.groupBy(F.to_date("tpep_pickup_datetime").alias("pickup_date")) \
    .count() \
    .withColumnRenamed("count", "nb_trips")

print("Busiest days")
trips_per_day.orderBy(F.col("nb_trips").desc()).show(3)

print("Slowest days")
trips_per_day.orderBy(F.col("nb_trips").asc()).show(3)

Busiest days
+-----------+--------+
|pickup_date|nb_trips|
+-----------+--------+
| 2019-01-25|  292499|
| 2019-01-11|  291714|
| 2019-01-31|  284625|
+-----------+--------+
only showing top 3 rows
Slowest days
+-----------+--------+
|pickup_date|nb_trips|
+-----------+--------+
| 2019-01-01|  189432|
| 2019-01-21|  192826|
| 2019-01-02|  198737|
+-----------+--------+
only showing top 3 rows


In [10]:
# Q6a: number of trips per hour of the day
trips_per_hour = df_jan.groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour")) \
    .count() \
    .withColumnRenamed("count", "nb_trips")

print("Trips per hour")
trips_per_hour.orderBy("pickup_hour").show(24)

print("Busiest hour")
trips_per_hour.orderBy(F.col("nb_trips").desc()).show(1)

print("Slowest hour")
trips_per_hour.orderBy(F.col("nb_trips").asc()).show(1)

# Q6b: number of trips per time of day
trips_per_period = df_jan.withColumn(
    "time_of_day",
    F.when(F.hour("tpep_pickup_datetime").between(6, 11), "morning")
     .when(F.hour("tpep_pickup_datetime").between(12, 17), "afternoon")
     .when(F.hour("tpep_pickup_datetime").between(18, 23), "evening")
     .otherwise("late night")
).groupBy("time_of_day").count().withColumnRenamed("count", "nb_trips")

print("Trips per time of day")
trips_per_period.orderBy(F.col("nb_trips").desc()).show()

Trips per hour
+-----------+--------+
|pickup_hour|nb_trips|
+-----------+--------+
|          0|  207758|
|          1|  149242|
|          2|  109413|
|          3|   78084|
|          4|   61423|
|          5|   75532|
|          6|  178598|
|          7|  304858|
|          8|  373735|
|          9|  365924|
|         10|  361382|
|         11|  375438|
|         12|  401172|
|         13|  404149|
|         14|  433115|
|         15|  452679|
|         16|  420806|
|         17|  468407|
|         18|  515374|
|         19|  475152|
|         20|  423128|
|         21|  409873|
|         22|  369026|
|         23|  281812|
+-----------+--------+

Busiest hour
+-----------+--------+
|pickup_hour|nb_trips|
+-----------+--------+
|         18|  515374|
+-----------+--------+
only showing top 1 row
Slowest hour
+-----------+--------+
|pickup_hour|nb_trips|
+-----------+--------+
|          4|   61423|
+-----------+--------+
only showing top 1 row
Trips per time of day
+-----------+---

In [11]:
from pyspark.sql import Window

# Q7: average number of trips per day of the week (January 2019)
trips_per_weekday = df_jan \
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE")) \
    .groupBy("day_of_week") \
    .agg(
        F.count("*").alias("total_trips"),                  # total trips for this weekday
        F.countDistinct("pickup_date").alias("nb_days")     # how many of this weekday in January
    ) \
    .withColumn("avg_trips_per_day", F.round(F.col("total_trips") / F.col("nb_days"), 0)) \
    .withColumn("rank", F.row_number().over(Window.orderBy(F.col("avg_trips_per_day").desc())))

# ranking from busiest (rank 1) to slowest (rank 7)
trips_per_weekday.orderBy("rank") \
    .select("rank", "day_of_week", "nb_days", "total_trips", "avg_trips_per_day") \
    .show()

+----+-----------+-------+-----------+-----------------+
|rank|day_of_week|nb_days|total_trips|avg_trips_per_day|
+----+-----------+-------+-----------+-----------------+
|   1|     Friday|      4|    1087150|         271788.0|
|   2|   Thursday|      5|    1356992|         271398.0|
|   3|  Wednesday|      5|    1265229|         253046.0|
|   4|   Saturday|      4|    1009979|         252495.0|
|   5|    Tuesday|      5|    1209076|         241815.0|
|   6|     Monday|      4|     907764|         226941.0|
|   7|     Sunday|      4|     859890|         214973.0|
+----+-----------+-------+-----------+-----------------+



In [12]:
# Q8: keep card payments only (cash tips are not recorded) and realistic values
df_tips = df_jan.filter(
    (F.col("payment_type") == 1) &
    (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) &
    (F.col("passenger_count").between(1, 6)) &
    (F.col("tip_amount") >= 0)
)

# correlation: close to 1 = strong link, close to 0 = no link
print("Correlation with tip amount")
df_tips.agg(
    F.round(F.corr("trip_distance", "tip_amount"), 3).alias("corr_distance_tip"),
    F.round(F.corr("passenger_count", "tip_amount"), 3).alias("corr_passengers_tip")
).show()

# average tip by distance range
print("Average tip by distance")
df_tips.withColumn(
    "distance_range",
    F.when(F.col("trip_distance") < 2, "1. < 2 miles")
     .when(F.col("trip_distance") < 5, "2. 2-5 miles")
     .when(F.col("trip_distance") < 10, "3. 5-10 miles")
     .otherwise("4. 10+ miles")
).groupBy("distance_range") \
 .agg(F.round(F.avg("tip_amount"), 2).alias("avg_tip"), F.count("*").alias("nb_trips")) \
 .orderBy("distance_range") \
 .show()

# average tip by passenger count
print("Average tip by passenger count")
df_tips.groupBy(F.col("passenger_count").cast("int").alias("passengers")) \
 .agg(F.round(F.avg("tip_amount"), 2).alias("avg_tip"), F.count("*").alias("nb_trips")) \
 .orderBy("passengers") \
 .show()

Correlation with tip amount
+-----------------+-------------------+
|corr_distance_tip|corr_passengers_tip|
+-----------------+-------------------+
|            0.708|              0.011|
+-----------------+-------------------+

Average tip by distance
+--------------+-------+--------+
|distance_range|avg_tip|nb_trips|
+--------------+-------+--------+
|  1. < 2 miles|   1.61| 3220297|
|  2. 2-5 miles|   2.66| 1409091|
| 3. 5-10 miles|   4.68|  427843|
|  4. 10+ miles|   8.39|  322389|
+--------------+-------+--------+

Average tip by passenger count
+----------+-------+--------+
|passengers|avg_tip|nb_trips|
+----------+-------+--------+
|         1|   2.52| 3919012|
|         2|   2.59|  777906|
|         3|   2.57|  217732|
|         4|   2.57|   91624|
|         5|   2.61|  230728|
|         6|    2.6|  142618|
+----------+-------+--------+



## Q8 Explanation – Does trip distance or passenger count affect the tip amount?

We only kept card payments, because cash tips are not recorded in the data (they show as $0). We also removed unrealistic values (distance of 0 or over 100 miles, 0 or more than 6 passengers).

**Distance: yes.** The correlation between distance and tip is 0.708, which is a strong link. 

This makes sense, since people usually tip a percentage of the fare, and the fare depends on the distance.

**Passenger count: no.** The correlation is 0.011, so there is almost no link. The average tip stays around $2.50-$2.60 whether there is 1 passenger or 6.

In [13]:
# Q9: trips with the highest extra charge
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "extra",
    "fare_amount",
    "trip_distance",
    "total_amount"
).orderBy(F.col("extra").desc()).show(5)

+-----------+--------------------+------+-----------+-------------+------------+
|    trip_id|tpep_pickup_datetime| extra|fare_amount|trip_distance|total_amount|
+-----------+--------------------+------+-----------+-------------+------------+
|25775127259| 2019-01-23 08:58:09|535.38|  355676.98|          0.0|   356214.78|
|25777257006| 2019-01-31 10:06:09| 23.04|        4.5|          0.0|       28.34|
|25770114828| 2019-01-02 16:33:28|  18.5|       52.0|        17.23|        88.3|
|25772258862| 2019-01-11 16:08:48|  18.5|       49.0|        15.78|       96.36|
|25769938325| 2019-01-01 16:09:32|  18.5|       39.5|         13.4|        70.8|
+-----------+--------------------+------+-----------+-------------+------------+
only showing top 5 rows


In [14]:
# Q10: count suspicious records in the whole dataset
checks = {
    "pickup outside January 2019": ~((F.col("tpep_pickup_datetime") >= "2019-01-01") & (F.col("tpep_pickup_datetime") < "2019-02-01")),
    "distance = 0": F.col("trip_distance") == 0,
    "distance > 100 miles": F.col("trip_distance") > 100,
    "duration <= 0 min": F.col("trip_duration_min") <= 0,
    "duration > 24 hours": F.col("trip_duration_min") > 24 * 60,
    "passenger count = 0": F.col("passenger_count") == 0,
    "passenger count > 6": F.col("passenger_count") > 6,
    "negative fare": F.col("fare_amount") < 0,
    "negative total amount": F.col("total_amount") < 0,
}

results = [(name, df_trips.filter(condition).count()) for name, condition in checks.items()]
spark.createDataFrame(results, ["anomaly", "nb_trips"]).show(truncate=False)

+---------------------------+--------+
|anomaly                    |nb_trips|
+---------------------------+--------+
|pickup outside January 2019|537     |
|distance = 0               |55089   |
|distance > 100 miles       |32      |
|duration <= 0 min          |6557    |
|duration > 24 hours        |5       |
|passenger count = 0        |117381  |
|passenger count > 6        |57      |
|negative fare              |7129    |
|negative total amount      |7127    |
+---------------------------+--------+



### Q10 Explanation – Outliers and strange data points

We checked each column against what is realistic for a NYC yellow taxi trip. The dataset has 7,696,617 trips.

- **0 passengers (117,381 trips)**: a trip can't have no passengers. The driver probably didn't enter the number.
- **More than 6 passengers (57 trips)**: a yellow cab takes 5 passengers max (6 with a child). Most trips with 9 passengers also have a distance of 0, so these are input errors.
- **Distance = 0 (55,089 trips)**: some still have a fare (e.g. $110.76). Probably cancelled trips or GPS/meter errors.
- **Distance > 100 miles (32 trips)**: some are impossible, like 831.8 miles in 9.5 minutes for $11.76.
- **Duration <= 0 (6,557 trips)**: the dropoff is before or at the same time as the pickup, so the timestamps are wrong.
- **Duration > 24 hours (5 trips)**: the meter was probably not stopped.
- **Negative fare or total (~7,100 trips)**: likely refunds or corrections, not real trips.
- **Pickup outside January 2019 (537 trips)**: some trips are dated November or December 2018, probably a clock error.
- **Extra and fare**: one trip has an extra of $535.38 and a fare of $355,676.98 with 0 miles, which is clearly an error.

These records are a small part of the data, but they can distort averages and min/max values. We didn't delete them, but we excluded them from the average-based analyses and kept only January 2019 pickups for the day/hour questions.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [15]:
# download the taxi zone lookup table
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_file = "taxi_zone_lookup.csv"

response = requests.get(zone_url)
if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

# create the dataframe (csv: header and schema must be specified)
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_file)

df_zones.printSchema()
df_zones.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [16]:
# zones for pickup and dropoff (renamed to avoid column name conflicts)
pu_zones = df_zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("pickup_borough"))
do_zones = df_zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("dropoff_borough"))

# join trips with pickup and dropoff boroughs
df_boroughs = df_jan \
    .join(pu_zones, on="PULocationID", how="left") \
    .join(do_zones, on="DOLocationID", how="left")

df_boroughs.cache()
df_boroughs.select("trip_id", "PULocationID", "pickup_borough", "DOLocationID", "dropoff_borough").show(5)

+-----------+------------+--------------+------------+---------------+
|    trip_id|PULocationID|pickup_borough|DOLocationID|dropoff_borough|
+-----------+------------+--------------+------------+---------------+
|25769803776|         151|     Manhattan|         239|      Manhattan|
|25769803777|         239|     Manhattan|         246|      Manhattan|
|25769803783|         163|     Manhattan|         229|      Manhattan|
|25769803784|         229|     Manhattan|           7|         Queens|
|25769803785|         141|     Manhattan|         234|      Manhattan|
+-----------+------------+--------------+------------+---------------+
only showing top 5 rows


In [17]:
# Q1: number of pickups and dropoffs by borough
print("Pickups by borough")
df_boroughs.groupBy("pickup_borough").agg(F.count("*").alias("nb_pickups")) \
    .orderBy(F.col("nb_pickups").desc()).show()

print("Dropoffs by borough")
df_boroughs.groupBy("dropoff_borough").agg(F.count("*").alias("nb_dropoffs")) \
    .orderBy(F.col("nb_dropoffs").desc()).show()

Pickups by borough
+--------------+----------+
|pickup_borough|nb_pickups|
+--------------+----------+
|     Manhattan|   6950511|
|        Queens|    471113|
|       Unknown|    159807|
|      Brooklyn|     91896|
|         Bronx|     18056|
|           N/A|      3890|
|           EWR|       446|
| Staten Island|       361|
+--------------+----------+

Dropoffs by borough
+---------------+-----------+
|dropoff_borough|nb_dropoffs|
+---------------+-----------+
|      Manhattan|    6816936|
|         Queens|     340914|
|       Brooklyn|     301074|
|        Unknown|     149091|
|          Bronx|      58068|
|            N/A|      16900|
|            EWR|      10913|
|  Staten Island|       2184|
+---------------+-----------+



In [18]:
from pyspark.sql import Window

# Q2: number of trips per borough and hour
trips_borough_hour = df_boroughs \
    .groupBy("pickup_borough", F.hour("tpep_pickup_datetime").alias("pickup_hour")) \
    .agg(F.count("*").alias("nb_trips"))

# rank hours inside each borough (busiest and slowest)
w_busy = Window.partitionBy("pickup_borough").orderBy(F.col("nb_trips").desc())
w_slow = Window.partitionBy("pickup_borough").orderBy(F.col("nb_trips").asc())

print("Busiest hour by borough")
trips_borough_hour.withColumn("rank", F.row_number().over(w_busy)) \
    .filter(F.col("rank") == 1).drop("rank") \
    .orderBy(F.col("nb_trips").desc()).show()

print("Slowest hour by borough")
trips_borough_hour.withColumn("rank", F.row_number().over(w_slow)) \
    .filter(F.col("rank") == 1).drop("rank") \
    .orderBy("pickup_borough").show()

Busiest hour by borough
+--------------+-----------+--------+
|pickup_borough|pickup_hour|nb_trips|
+--------------+-----------+--------+
|     Manhattan|         18|  471524|
|        Queens|         16|   29880|
|       Unknown|         18|   10751|
|      Brooklyn|          8|    6935|
|         Bronx|          7|    1803|
|           N/A|         19|     214|
|           EWR|         15|      54|
| Staten Island|          8|      36|
+--------------+-----------+--------+

Slowest hour by borough
+--------------+-----------+--------+
|pickup_borough|pickup_hour|nb_trips|
+--------------+-----------+--------+
|         Bronx|          3|     225|
|      Brooklyn|          3|    1919|
|           EWR|         23|       1|
|     Manhattan|          4|   53446|
|           N/A|          6|      88|
|        Queens|          3|    3084|
| Staten Island|          1|       3|
|       Unknown|          4|    1465|
+--------------+-----------+--------+



In [19]:
# Q3: average trips per day of the week, by borough
trips_borough_weekday = df_boroughs \
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE")) \
    .groupBy("pickup_borough", "day_of_week") \
    .agg(F.count("*").alias("total_trips"), F.countDistinct("pickup_date").alias("nb_days")) \
    .withColumn("avg_trips_per_day", F.round(F.col("total_trips") / F.col("nb_days"), 0))

w_day = Window.partitionBy("pickup_borough").orderBy(F.col("avg_trips_per_day").desc())

print("Busiest day of the week by borough")
trips_borough_weekday.withColumn("rank", F.row_number().over(w_day)) \
    .filter(F.col("rank") == 1) \
    .select("pickup_borough", "day_of_week", "avg_trips_per_day") \
    .orderBy(F.col("avg_trips_per_day").desc()).show()

Busiest day of the week by borough
+--------------+-----------+-----------------+
|pickup_borough|day_of_week|avg_trips_per_day|
+--------------+-----------+-----------------+
|     Manhattan|     Friday|         246224.0|
|        Queens|     Monday|          16662.0|
|       Unknown|   Thursday|           5785.0|
|      Brooklyn|     Friday|           3273.0|
|         Bronx|     Friday|            667.0|
|           N/A|    Tuesday|            141.0|
|           EWR|     Friday|             19.0|
| Staten Island|     Friday|             16.0|
+--------------+-----------+-----------------+



In [20]:
# clean data for averages (outliers found in part 1 are removed)
df_clean = df_boroughs.filter(
    (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) &
    (F.col("fare_amount") > 0)
)

# Q4 & Q5: average trip distance and fare by pickup borough
df_clean.groupBy("pickup_borough") \
    .agg(
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.count("*").alias("nb_trips")
    ) \
    .orderBy(F.col("avg_distance_miles").desc()) \
    .show()

+--------------+------------------+--------+--------+
|pickup_borough|avg_distance_miles|avg_fare|nb_trips|
+--------------+------------------+--------+--------+
| Staten Island|             13.91|   44.68|     323|
|        Queens|             11.61|   35.71|  457391|
|           EWR|              7.69|   78.26|     138|
|         Bronx|              7.57|   26.94|   17208|
|           N/A|              5.28|   40.13|    2101|
|      Brooklyn|              4.91|   18.84|   89474|
|       Unknown|              2.55|   11.78|  151542|
|     Manhattan|              2.24|   10.72| 6916065|
+--------------+------------------+--------+--------+



In [ ]:
# Q6: highest and lowest fare amounts with their boroughs
cols_fare = ["trip_id", "fare_amount", "trip_distance", "pickup_borough", "dropoff_borough"]

print("Highest fares")
df_boroughs.orderBy(F.col("fare_amount").desc()).select(cols_fare).show(5)

print("Lowest fares")
df_boroughs.orderBy(F.col("fare_amount").asc()).select(cols_fare).show(5)

Highest fares


In [ ]:
# Q7: download the most recent available January dataset
for year in [2026, 2025]:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-01.parquet"
    response = requests.get(url)
    if response.status_code == 200:
        recent_file = f"yellow_tripdata_{year}-01.parquet"
        with open(recent_file, "wb") as f:
            f.write(response.content)
        recent_year = year
        print("Loaded January", year)
        break

# keep only pickups in January of that year
df_recent = spark.read.parquet(recent_file).filter(
    (F.col("tpep_pickup_datetime") >= f"{recent_year}-01-01") &
    (F.col("tpep_pickup_datetime") < f"{recent_year}-02-01")
)

In [ ]:
# compute the same average metrics for a given dataframe
def average_metrics(df, label):
    return df.filter(
        (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) & (F.col("fare_amount") > 0)
    ).agg(
        F.lit(label).alias("dataset"),
        F.count("*").alias("nb_trips"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60), 1).alias("avg_duration_min"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("total_amount"), 2).alias("avg_total")
    )

# compare January 2019 with the most recent January
average_metrics(df_jan, "January 2019") \
    .union(average_metrics(df_recent, f"January {recent_year}")) \
    .show()

### Q7 – January 2019 vs January 2026

Yes, the averages changed. In 2026 there are about half as many trips, with fewer passengers per trip (1.25 vs 1.57). Trips are a bit longer (3.48 vs 2.85 miles) but take about the same time. The main change is the price: the average total almost doubled (29.62 dollars vs 15.66 dollars), and tips went up too (2.68 dollars vs 1.81 dollars).

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [ ]:
# register the dataframes as temporary SQL views
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [ ]:
# Average passenger count
spark.sql("""
    SELECT AVG(passenger_count) AS avg_passenger_count
    FROM trips
""").show()

In [ ]:
# Busiest day
spark.sql("""
    SELECT TO_DATE(tpep_pickup_datetime) AS pickup_date, COUNT(*) AS trip_count
    FROM trips
    WHERE tpep_pickup_datetime >= '2019-01-01'
      AND tpep_pickup_datetime <  '2019-02-01'
    GROUP BY TO_DATE(tpep_pickup_datetime)
    ORDER BY trip_count DESC
    LIMIT 1
""").show()

# Slowest day
spark.sql("""
    SELECT TO_DATE(tpep_pickup_datetime) AS pickup_date, COUNT(*) AS trip_count
    FROM trips
    WHERE tpep_pickup_datetime >= '2019-01-01'
      AND tpep_pickup_datetime <  '2019-02-01'
    GROUP BY TO_DATE(tpep_pickup_datetime)
    ORDER BY trip_count ASC
    LIMIT 1
""").show()

In [ ]:
# Join trips with zones to get pickup borough, then compute average distance per borough
print("Average trip distance by pickup borough (January 2019)")
spark.sql("""
    SELECT z.Borough AS pickup_borough,
           ROUND(AVG(t.trip_distance), 2) AS avg_trip_distance,
           COUNT(*) AS nb_trips
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    WHERE t.tpep_pickup_datetime >= '2019-01-01'
      AND t.tpep_pickup_datetime <  '2019-02-01'
      AND t.trip_distance > 0
      AND t.trip_distance < 100
      AND t.fare_amount > 0
    GROUP BY z.Borough
    ORDER BY avg_trip_distance DESC
""").show()

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

In [ ]:
import os

# download every month of 2019 (skip files already downloaded)
files_2019 = []
for month in range(1, 13):
    file_name = f"yellow_tripdata_2019-{month:02d}.parquet"
    if not os.path.exists(file_name):
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file_name}"
        response = requests.get(url)
        if response.status_code == 200:
            with open(file_name, "wb") as f:
                f.write(response.content)
    files_2019.append(file_name)
    print("Ready:", file_name)

In [ ]:
from functools import reduce

# read each month, keep only the pickup time (column types can differ between files)
df_2019 = reduce(
    lambda a, b: a.union(b),
    [spark.read.parquet(f).select("tpep_pickup_datetime") for f in files_2019]
).filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01") &
    (F.col("tpep_pickup_datetime") < "2020-01-01")
)

# assign a season to each trip based on the month
df_seasons = df_2019.withColumn(
    "season",
    F.when(F.month("tpep_pickup_datetime").isin(12, 1, 2), "winter")
     .when(F.month("tpep_pickup_datetime").isin(3, 4, 5), "spring")
     .when(F.month("tpep_pickup_datetime").isin(6, 7, 8), "summer")
     .otherwise("fall")
)

# average trips per day for each season (seasons don't have the same number of days)
trips_per_season = df_seasons \
    .groupBy("season") \
    .agg(
        F.count("*").alias("total_trips"),
        F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("nb_days")
    ) \
    .withColumn("avg_trips_per_day", F.round(F.col("total_trips") / F.col("nb_days"), 0)) \
    .orderBy(F.col("avg_trips_per_day").desc())

trips_per_season.show()

In [ ]:
%pip install plotly #if not installed 

In [ ]:
# Viz: average trips per day by season (2019)
fig = trips_per_season.plot.bar(x="season", y="avg_trips_per_day")
fig.update_layout(title="Average number of trips per day by season (2019)")
fig.show()

### Where to go from here Explanation (bonus)

We loaded all the 2019 data to find the busiest season (see the `trips_per_season` table above).

For the visualizations, we tried to use Spark 4's native plotting (`df.plot.bar(...)`), but it needs the plotly library, which isn't installed in the Docker image. We tried `%pip install plotly` but didn't manage to get the charts working in time.

We would have plotted the average trips per weekday, trips per hour, and pickups by borough. Another option would have been to use `.toPandas()` with matplotlib.